Лабораторная работа №1

Решите следующие задачи для данных велопарковок Сан-Франциско (trips.csv, stations.csv):

Найти велосипед с максимальным временем пробега.

Найти наибольшее геодезическое расстояние между станциями.

Найти путь велосипеда с максимальным временем пробега через станции.

Найти количество велосипедов в системе.

Найти пользователей потративших на поездки более 3 часов.

In [11]:
from pyspark.sql import SparkSession

import pyspark.sql.functions as F
import pyspark.sql.types as t
from geopy.distance import geodesic
from math import sqrt
import pandas as pd

import os; os.environ['JAVA_HOME'] = r'C:\Java\LibericaJDK-11-Full'; os.environ['HADOOP_HOME'] = r'C:\hadoop'; os.environ['PATH'] = os.environ['JAVA_HOME'] + r'\bin;' + os.environ['HADOOP_HOME'] + r'\bin;' + os.environ['PATH']

In [12]:
spark = SparkSession.builder.getOrCreate()
spark

In [22]:
trips_path = 'file:///C:/Users/aniam/Documents/8_sem/Bigdata/LR_1/trips.csv'
stations_path = 'file:///C:/Users/aniam/Documents/8_sem/Bigdata/LR_1/stations.csv'

data_trips = spark.read.format('csv').option('header', 'true').load(trips_path)
data_stations = spark.read.format('csv').option('header', 'true').load(stations_path)

data_trips.show(5)
data_stations.show(5)

+----+--------+---------------+--------------------+----------------+---------------+--------------------+--------------+-------+-----------------+--------+
|  id|duration|     start_date|  start_station_name|start_station_id|       end_date|    end_station_name|end_station_id|bike_id|subscription_type|zip_code|
+----+--------+---------------+--------------------+----------------+---------------+--------------------+--------------+-------+-----------------+--------+
|4576|      63|8/29/2013 14:13|South Van Ness at...|              66|8/29/2013 14:14|South Van Ness at...|            66|    520|       Subscriber|   94127|
|4607|      70|8/29/2013 14:42|  San Jose City Hall|              10|8/29/2013 14:43|  San Jose City Hall|            10|    661|       Subscriber|   95138|
|4130|      71|8/29/2013 10:16|Mountain View Cit...|              27|8/29/2013 10:17|Mountain View Cit...|            27|     48|       Subscriber|   97214|
|4251|      77|8/29/2013 11:29|  San Jose City Hall|      

Найти велосипед с максимальным временем пробега.

In [ ]:
bike_total_duration = data_trips.groupBy('bike_id') \
    .agg(F.sum('duration').alias('total_duration'))

max_bike = bike_total_duration.orderBy(F.desc('total_duration')).limit(1)

max_bike.show()

+-------+--------------+
|bike_id|total_duration|
+-------+--------------+
|    535|   1.8611693E7|
+-------+--------------+



Найти наибольшее геодезическое расстояние между станциями.

In [31]:
import math
from pyspark.sql import functions as F

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    d_phi = math.radians(lat2 - lat1)
    d_lambda = math.radians(lon2 - lon1)
    a = math.sin(d_phi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(d_lambda/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    return R * c

data_stations = data_stations \
    .withColumn('lat', F.col('lat').cast('double')) \
    .withColumn('long', F.col('long').cast('double')) \
    .filter(F.col('lat').isNotNull() & F.col('long').isNotNull())

coords_list = data_stations.select('id', 'name', 'lat', 'long').collect()

max_dist = 0
max_pair = None

for i in range(len(coords_list)):
    for j in range(i+1, len(coords_list)):
        s1 = coords_list[i]
        s2 = coords_list[j]
        dist = haversine(s1['lat'], s1['long'], s2['lat'], s2['long'])
        if dist > max_dist:
            max_dist = dist
            max_pair = (s1, s2)

print(f"Максимальное расстояние: {max_dist:.2f} км")
print(f"Между: {max_pair[0]['name']} и {max_pair[1]['name']}")

Максимальное расстояние: 69.92 км
Между: SJSU - San Salvador at 9th и Embarcadero at Sansome


Найти путь велосипеда с максимальным временем пробега через станции.

In [33]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bike_total_duration = data_trips.groupBy('bike_id') \
    .agg(F.sum('duration').alias('total_duration'))

max_bike_id = bike_total_duration.orderBy(F.desc('total_duration')).limit(1).collect()[0]['bike_id']
print(f"Велосипед ID: {max_bike_id}")

bike_trips = data_trips.filter(F.col('bike_id') == max_bike_id) \
    .withColumn('start_date_ts', F.to_timestamp('start_date', 'M/d/yyyy H:mm')) \
    .orderBy('start_date_ts')

print(f"\nПуть велосипеда #{max_bike_id} ({bike_trips.count()} поездок):\n")

bike_trips.select(
    'start_date',
    'start_station_name',
    'end_station_name',
    'duration'
).show(truncate=False)

Велосипед ID: 535

Путь велосипеда #535 (1328 поездок):

+---------------+---------------------------------------------+----------------------------------------+--------+
|start_date     |start_station_name                           |end_station_name                        |duration|
+---------------+---------------------------------------------+----------------------------------------+--------+
|8/29/2013 19:32|Post at Kearney                              |San Francisco Caltrain (Townsend at 4th)|1245    |
|8/29/2013 21:38|San Francisco Caltrain (Townsend at 4th)     |San Francisco Caltrain 2 (330 Townsend) |423     |
|8/30/2013 8:40 |San Francisco Caltrain 2 (330 Townsend)      |Market at Sansome                       |842     |
|8/30/2013 9:10 |Market at Sansome                            |2nd at South Park                       |498     |
|9/1/2013 12:58 |2nd at Townsend                              |Davis at Jackson                        |1671    |
|9/5/2013 11:59 |San Francisco 

Найти количество велосипедов в системе

In [34]:
bike_count = data_trips.select('bike_id').distinct().count()

print(f"Всего велосипедов в системе: {bike_count}")

Всего велосипедов в системе: 700


Найти пользователей потративших на поездки более 3 часов

In [41]:
user_total_time = data_trips.groupBy('zip_code') \
    .agg(
        F.sum('duration').alias('total_minutes'),
        F.count('id').alias('trips_count')
    ) \
    .filter(F.col('total_minutes') > 180)

result = user_total_time.orderBy(F.desc('total_minutes'))

print(f"Пользователи с общим временем поездок > 3 часов:\n")
result.select(
    'zip_code',
    F.col('total_minutes').cast('int').alias('total_minutes'),
    F.round(F.col('total_minutes').cast('double') / 60.0, 2).alias('total_hours'),
    'trips_count'
).orderBy(F.desc('total_minutes')).show()

Пользователи с общим временем поездок > 3 часов:

+--------+-------------+-----------+-----------+
|zip_code|total_minutes|total_hours|trips_count|
+--------+-------------+-----------+-----------+
|   94107|     49757162|  829286.03|      78704|
|     nil|     45725550|   762092.5|      10682|
|    NULL|     27723273|  462054.55|       6619|
|   94105|     25596128|  426602.13|      42672|
|   94133|     21637675|  360627.92|      31359|
|   94102|     19128021|  318800.35|      19757|
|   94103|     19127388|   318789.8|      26673|
|   95531|     17270400|   287840.0|          1|
|   94111|     14244997|  237416.62|      21409|
|   95112|     12742370|  212372.83|      11564|
|   94109|     12057128|  200952.13|      13989|
|   94040|      7807926|   130132.1|       7114|
|   94110|      7421936|  123698.93|       7621|
|   94117|      6901313|  115021.88|       9851|
|   94301|      6590378|  109839.63|       3868|
|   94041|      6276284|  104604.73|       5867|
|   94158|      624